# torch-spatiotemporal (tsl) — Getting Started

This notebook walks through the core tsl workflow using the built-in **MetrLA** traffic dataset.  
The same pattern applies when we swap in USGS streamflow gauge data.

**Workflow:**
1. Load a built-in dataset
2. Inspect the graph structure and time series
3. Wrap data in `SpatioTemporalDataset` (handles windowing + batching)
4. Build a DataModule for Lightning
5. Define a spatiotemporal model
6. Train with `Predictor` + PyTorch Lightning
7. Evaluate predictions

## 1. Load the MetrLA Dataset

MetrLA: **207 traffic sensors** in LA at **5-min resolution** over ~4 months.  
tsl ships it ready-to-use — downloads automatically on first run.

In [1]:
from tsl.datasets import MetrLA

: 

## 2. Inspect the Graph Structure

tsl uses **COO (edge list) format** for connectivity — same convention as PyTorch Geometric.  
MetrLA builds edges from a Gaussian kernel on road distances between sensors.

In [ ]:
import numpy as np

# get_connectivity() builds the sparse graph
#   threshold    : drop edges with weight below this value (sparsify)
#   include_self : add self-loops
#   normalize_axis=1 : row-normalize so edge weights sum to 1 per node
#   layout='edge_index' : returns (edge_index, edge_weight) in COO format
connectivity = dataset.get_connectivity(
    threshold=0.1,
    include_self=False,
    normalize_axis=1,
    layout='edge_index'
)

edge_index, edge_weight = connectivity

n_nodes = df.shape[1]
print(f'Nodes  : {n_nodes}')
print(f'Edges  : {edge_index.shape[1]}')
print(f'edge_index shape : {edge_index.shape}  <- (2, E): row0=src, row1=dst')
print(f'Weight range     : [{edge_weight.min():.3f}, {edge_weight.max():.3f}]')

In [ ]:
import matplotlib.pyplot as plt

# Quick look at the time series for 5 nodes
fig, ax = plt.subplots(figsize=(12, 3))
df.iloc[:500, :5].plot(ax=ax, alpha=0.8)
ax.set_title('Traffic speed — first 500 timesteps, 5 nodes')
ax.set_ylabel('Speed (mph)')
plt.tight_layout()
plt.show()

## 3. Build a `SpatioTemporalDataset`

`SpatioTemporalDataset` is the heart of tsl. It:
- Stores the node time series as a tensor `(T, N, C)`
- Holds graph connectivity
- Slices data into sliding **input windows** → **forecast horizons** on-the-fly
- Returns `StaticBatch` objects per sample (like PyG `Data`, but with time dims)

Key arguments:
- `window`  — number of past timesteps fed as input
- `horizon` — number of future timesteps to predict
- `stride`  — step size between consecutive samples

In [ ]:
from tsl.data import SpatioTemporalDataset

WINDOW  = 12   # 12 steps x 5 min = 1 hour of context
HORIZON = 12   # predict the next 1 hour

torch_dataset = SpatioTemporalDataset(
    target=df,               # (T, N) DataFrame — tsl adds a channel dim -> (T, N, 1)
    connectivity=connectivity,
    window=WINDOW,
    horizon=HORIZON,
    stride=1,                # slide the window by 1 step each sample
)

print('Number of samples:', len(torch_dataset))

# Inspect one sample
sample = torch_dataset[0]
print('\nKeys in sample.input:', list(sample.input.keys()))
print('Input  x : (window, N, C) =', sample.input.x.shape)
print('Target y : (horizon, N, C) =', sample.y.shape)

## 4. DataModule — Train / Val / Test Splits

`SpatioTemporalDataModule` wraps the dataset into a Lightning `DataModule`.  
It handles:
- Chronological train/val/test splits (no shuffling across time)
- Fitting a `StandardScaler` **only on the training set** (no data leakage)
- Creating PyTorch DataLoaders

In [ ]:
from tsl.data import SpatioTemporalDataModule
from tsl.data.preprocessing import StandardScaler

# Normalize each node to zero mean, unit variance
# axis=(0,1) computes statistics over both time AND node dimensions
scaler = StandardScaler(axis=(0, 1))

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers={'target': scaler},
    # temporal split: last 20% = test, preceding 10% = val, rest = train
    splitter=dataset.get_splitter(val_len=0.1, test_len=0.2),
    batch_size=32,
    workers=0,
)

dm.setup()
print('Train batches:', len(dm.train_dataloader()))
print('Val   batches:', len(dm.val_dataloader()))
print('Test  batches:', len(dm.test_dataloader()))

# Confirm batch shapes
batch = next(iter(dm.train_dataloader()))
print('\nBatch input x :', batch.input.x.shape)  # (B, window, N, C)
print('Batch target y :', batch.y.shape)         # (B, horizon, N, C)

## 5. Define a Spatiotemporal Model

We use **`TimeThenSpaceModel`**, a strong two-stage baseline:
1. **Temporal encoder** — processes each node's time series independently (TCN here)
2. **Spatial encoder (GNN)** — mixes node representations across the graph
3. **Decoder** — projects to forecast horizon

Other available models: `STConvNet`, `GRINModel`, `DCRNNModel`, `TransformerModel`.

In [ ]:
from tsl.nn.models import TimeThenSpaceModel

model = TimeThenSpaceModel(
    input_size=1,        # C — features per node per timestep (just speed here)
    n_nodes=n_nodes,     # N — number of graph nodes
    horizon=HORIZON,     # steps to forecast
    output_size=1,       # C_out — one output per node per step
    # Temporal encoder: Temporal Convolutional Network
    # Captures patterns at multiple time scales via dilated convolutions
    temporal_encoder_class='TCNEncoder',
    temporal_encoder_kwargs=dict(
        hidden_channels=32,
        kernel_size=2,
        n_layers=4,       # 4 dilated layers -> receptive field = 2^4 = 16 steps
    ),
    # Spatial encoder: Graph Attention Network
    # Learns to weight neighbor contributions based on learned attention scores
    spatial_encoder_class='GATConv',
    spatial_encoder_kwargs=dict(
        out_channels=32,
        heads=4,
    ),
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

## 6. Train with `Predictor` + PyTorch Lightning

`Predictor` is tsl's Lightning module. It adds:
- Loss function + optimizer
- Automatic inverse-scaling of predictions before computing metrics
- Built-in logging of MAE / MSE / MAPE per epoch

In [ ]:
import torch
import pytorch_lightning as pl
from tsl.engines import Predictor
from tsl.metrics.torch import MaskedMAE, MaskedMSE

predictor = Predictor(
    model=model,
    optim_class=torch.optim.Adam,
    optim_kwargs={'lr': 1e-3},
    loss_fn=MaskedMAE(),          # MAE that ignores NaN-masked entries
    metrics={'mae': MaskedMAE(), 'mse': MaskedMSE()},
    scale_target=True,            # inverse-scale preds before reporting metrics
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',   # auto-selects GPU if available
    devices=1,
)

trainer.fit(predictor, datamodule=dm)

## 7. Evaluate on Test Set

In [ ]:
results = trainer.test(predictor, datamodule=dm)
print(results)

In [ ]:
import torch
import matplotlib.pyplot as plt

predictor.eval()
batch = next(iter(dm.test_dataloader()))

with torch.no_grad():
    y_hat = predictor(batch.input)   # forward pass -> (B, horizon, N, 1)

# Pick one sample and one node to visualize
node = 0
y_true = batch.y[0, :, node, 0].cpu().numpy()
y_pred = y_hat[0, :, node, 0].cpu().numpy()

# Inverse-scale back to original units (mph)
y_true = dm.scalers['target'].inverse_transform(y_true)
y_pred = dm.scalers['target'].inverse_transform(y_pred)

plt.figure(figsize=(8, 3))
plt.plot(y_true, label='True', marker='o')
plt.plot(y_pred, label='Predicted', marker='x')
plt.title(f'Node {node} — 12-step (1 hr) forecast')
plt.xlabel('Horizon step (5 min each)')
plt.ylabel('Speed (mph)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Next Steps → USGS Streamflow

Adapting this pipeline to streamflow prediction requires only a few swaps:

| Component | MetrLA (here) | USGS Streamflow |
|---|---|---|
| `target` DataFrame | traffic speed (T×N) | discharge in cfs (T×N) |
| `connectivity` | road-distance Gaussian kernel | watershed adjacency or flow-distance |
| `window` / `horizon` | 12 steps × 5 min | e.g. 7 days input → 3 days forecast |
| Covariates | none | precipitation, temperature, basin attributes |

The `SpatioTemporalDataset` accepts extra node-level covariates via its `covariates` argument,
and most tsl models accept an `u` (exogenous) input tensor alongside `x`.